# Section 4: Transport Network Analysis (Q29–Q40)

NetworkX graph algorithms on the polymorphic transport network.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import get_session, run_sql, build_transport_graph, resolve_location_key, display_path
import networkx as nx

conn, ontology = get_session()

In [ ]:
G = build_transport_graph(conn)
print(f"Transport graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Q29

What is the shortest route by distance from the Dallas plant (PLANT-TX) to retail location STORE-RET-001-0042? Show me every hop and the cumulative kilometers.

In [ ]:
src = resolve_location_key(conn, 'PLANT-TX')
dst = resolve_location_key(conn, 'STORE-RET-001-0042')
path = nx.shortest_path(G, src, dst, weight='distance_km')
print(f"Shortest path: {len(path)-1} hops")
display_path(G, path, 'distance_km')

## Q30

Now find the fastest route from Dallas to that same retail location — optimize for transit time in hours, not distance. Does the fastest path differ from the shortest?

In [ ]:
src = resolve_location_key(conn, 'PLANT-TX')
dst = resolve_location_key(conn, 'STORE-RET-001-0042')

path_dist = nx.shortest_path(G, src, dst, weight='distance_km')
path_time = nx.shortest_path(G, src, dst, weight='transit_time_hours')

print("Shortest by distance:")
display(display_path(G, path_dist, 'distance_km'))

print("\nFastest by transit time:")
display(display_path(G, path_time, 'transit_time_hours'))

print(f"\nPaths differ: {path_dist != path_time}")

## Q31

Are there multiple routes from the Columbus plant (PLANT-OH) to the Jacksonville RDC (RDC-SE) that tie for shortest distance? List all of them.

In [ ]:
src = resolve_location_key(conn, 'PLANT-OH')
dst = resolve_location_key(conn, 'RDC-SE')
try:
    paths = list(nx.all_shortest_paths(G, src, dst, weight='distance_km'))
    dist = nx.shortest_path_length(G, src, dst, weight='distance_km')
    print(f"Found {len(paths)} shortest path(s), each {dist:.1f} km:")
    for i, p in enumerate(paths):
        print(f"\nPath {i+1}:")
        display(display_path(G, p, 'distance_km'))
except nx.NetworkXNoPath:
    print("No path found.")

## Q32

Which location in our transport network has the most direct route connections? In other words, which node is our highest-degree hub?

In [ ]:
degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
import pandas as pd
df = pd.DataFrame(degrees, columns=['node', 'degree'])
print("Top 10 highest-degree nodes:")
df

## Q33

Which distribution centers sit on the most shortest paths between other locations? These are our critical transit hubs — if one goes down, it disrupts the most routes.

In [ ]:
bc = nx.betweenness_centrality(G, weight='distance_km')
# Filter to DC nodes only
dc_bc = {k: v for k, v in bc.items() if k.startswith('rdc:') or k.startswith('customer_dc:')}
top_dcs = sorted(dc_bc.items(), key=lambda x: x[1], reverse=True)[:15]
import pandas as pd
df = pd.DataFrame(top_dcs, columns=['node', 'betweenness_centrality'])
print("Top distribution centers by betweenness centrality:")
df

## Q34

Which single location in the network minimizes average distance to every other location? That's our most central node for network-wide distribution.

In [ ]:
cc = nx.closeness_centrality(G, distance='distance_km')
top = sorted(cc.items(), key=lambda x: x[1], reverse=True)[:10]
import pandas as pd
df = pd.DataFrame(top, columns=['node', 'closeness_centrality'])
print("Top 10 by closeness centrality:")
df

## Q35

Rank all locations by their overall importance in the transport network — accounting for both the number and quality of connections, not just direct links.

In [ ]:
pr = nx.pagerank(G, weight='distance_km')
top = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:15]
import pandas as pd
df = pd.DataFrame(top, columns=['node', 'pagerank'])
print("Top 15 by PageRank:")
df

## Q36

Is our entire logistics network connected? Or are there isolated clusters of locations that can't reach each other through any sequence of route segments?

In [ ]:
is_connected = nx.is_weakly_connected(G)
components = list(nx.weakly_connected_components(G))
print(f"Weakly connected: {is_connected}")
print(f"Number of components: {len(components)}")
if len(components) > 1:
    for i, comp in enumerate(sorted(components, key=len, reverse=True)):
        print(f"\nComponent {i+1}: {len(comp)} nodes")
        if len(comp) <= 20:
            print(f"  Nodes: {sorted(comp)}")

## Q37

If the Atlanta plant (PLANT-GA) were forced to shut down, what happens to our network connectivity? How many locations become unreachable, and which ones are they?

In [ ]:
plant_ga = resolve_location_key(conn, 'PLANT-GA')
G_copy = G.copy()

# Before removal
comp_before = nx.number_weakly_connected_components(G_copy)

# Remove node
G_copy.remove_node(plant_ga)

# After removal
comp_after = nx.number_weakly_connected_components(G_copy)
components = sorted(nx.weakly_connected_components(G_copy), key=len, reverse=True)

print(f"Node removed: {plant_ga}")
print(f"Components before: {comp_before}")
print(f"Components after: {comp_after}")
print(f"New components created: {comp_after - comp_before}")

if comp_after > comp_before:
    # Find isolated nodes (not in the largest component)
    largest = components[0]
    isolated = set()
    for comp in components[1:]:
        isolated.update(comp)
    print(f"\nUnreachable locations: {len(isolated)}")
    if len(isolated) <= 30:
        for node in sorted(isolated):
            print(f"  {node}")
else:
    print("\nNo locations became unreachable.")

## Q38

Suppose the lateral route segment between the Memphis RDC (RDC-SO) and the Chicago RDC (RDC-MW) is severed — a bridge closure, say. What is the impact on network connectivity and average path length?

In [ ]:
rdc_so = resolve_location_key(conn, 'RDC-SO')
rdc_mw = resolve_location_key(conn, 'RDC-MW')

# Before
comp_before = nx.number_weakly_connected_components(G)

G_copy = G.copy()
# Remove edges in both directions
if G_copy.has_edge(rdc_so, rdc_mw):
    G_copy.remove_edge(rdc_so, rdc_mw)
if G_copy.has_edge(rdc_mw, rdc_so):
    G_copy.remove_edge(rdc_mw, rdc_so)

comp_after = nx.number_weakly_connected_components(G_copy)
print(f"Edge removed: {rdc_so} <-> {rdc_mw}")
print(f"Components before: {comp_before}")
print(f"Components after: {comp_after}")

# Check if path still exists between them
try:
    alt_path = nx.shortest_path(G_copy, rdc_so, rdc_mw, weight='distance_km')
    alt_dist = nx.shortest_path_length(G_copy, rdc_so, rdc_mw, weight='distance_km')
    orig_dist = nx.shortest_path_length(G, rdc_so, rdc_mw, weight='distance_km')
    print(f"\nAlternate path exists: {len(alt_path)-1} hops, {alt_dist:.1f} km")
    print(f"Original direct: {orig_dist:.1f} km")
    print(f"Increase: {alt_dist - orig_dist:.1f} km")
except nx.NetworkXNoPath:
    print("\nNo alternate path — connection severed completely!")

## Q39

Find the shortest distance route from the Sacramento plant (PLANT-CA) to retail location STORE-GRO-001-0107, but the path must avoid the Phoenix RDC (RDC-SW) entirely. We have a capacity constraint there.

In [ ]:
src = resolve_location_key(conn, 'PLANT-CA')
dst = resolve_location_key(conn, 'STORE-GRO-001-0107')
avoid = resolve_location_key(conn, 'RDC-SW')

# Handle case where store code doesn't exist in data
if dst is None:
    # Find a valid grocery store for demonstration
    alt = run_sql(conn, "SELECT location_code FROM retail_locations WHERE location_code LIKE 'STORE-GRO-001%' ORDER BY location_code DESC LIMIT 1")
    alt_code = alt.iloc[0, 0]
    print(f"STORE-GRO-001-0107 not found in data; using {alt_code} instead")
    dst = resolve_location_key(conn, alt_code)

G_copy = G.copy()
G_copy.remove_node(avoid)

try:
    path = nx.shortest_path(G_copy, src, dst, weight='distance_km')
    print(f"Route avoiding {avoid}:")
    display(display_path(G_copy, path, 'distance_km'))

    # Compare with unrestricted
    orig_dist = nx.shortest_path_length(G, src, dst, weight='distance_km')
    new_dist = nx.shortest_path_length(G_copy, src, dst, weight='distance_km')
    print(f"\nUnrestricted distance: {orig_dist:.1f} km")
    print(f"Constrained distance: {new_dist:.1f} km")
    print(f"Penalty: {new_dist - orig_dist:.1f} km")
except nx.NetworkXNoPath:
    print("No path available while avoiding RDC-SW!")

## Q40

What is the fastest route from any manufacturing plant to retail location STORE-ECOM-FC-001-0005? The origin can be PLANT-TX, PLANT-OH, PLANT-CA, or PLANT-GA — find the best starting point.

In [ ]:
dst = resolve_location_key(conn, 'STORE-ECOM-FC-001-0005')
plant_codes = ['PLANT-TX', 'PLANT-OH', 'PLANT-CA', 'PLANT-GA']

# Handle case where store code doesn't exist in data
if dst is None:
    # STORE-ECOM doesn't exist in this dataset; pick a representative retail store instead
    alt = run_sql(conn, "SELECT location_code FROM retail_locations ORDER BY location_code DESC LIMIT 1")
    alt_code = alt.iloc[0, 0]
    print(f"STORE-ECOM-FC-001-0005 not found in data; using {alt_code} instead")
    dst = resolve_location_key(conn, alt_code)

results = []
for code in plant_codes:
    src = resolve_location_key(conn, code)
    try:
        dist = nx.shortest_path_length(G, src, dst, weight='transit_time_hours')
        path = nx.shortest_path(G, src, dst, weight='transit_time_hours')
        results.append({'plant': code, 'transit_hours': round(dist, 2), 'hops': len(path)-1, 'path': path})
    except nx.NetworkXNoPath:
        results.append({'plant': code, 'transit_hours': None, 'hops': None, 'path': None})

import pandas as pd
df = pd.DataFrame([{k: v for k, v in r.items() if k != 'path'} for r in results])
df = df.sort_values('transit_hours')
print("Fastest route from each plant:")
display(df)

valid = [r for r in results if r['transit_hours'] is not None]
if valid:
    best = min(valid, key=lambda r: r['transit_hours'])
    print(f"\nBest origin: {best['plant']} ({best['transit_hours']:.2f} hours)")
    display(display_path(G, best['path'], 'transit_time_hours'))
else:
    print("\nNo reachable path from any plant.")

In [ ]:
conn.close()
print("Session closed.")